In [ ]:
import requests
import os

# URL de la API de Showdown para buscar replays recientes (filtrados por gen9ou)
url = "https://replay.pokemonshowdown.com/search.json?format=gen9ou"

# Carpeta donde se guardarán los archivos .log
output_folder = "../data/batallas_ou_logs"

# Crear carpeta si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

pages=100

for i in range(1,pages+1):
  # Hacer la solicitud para obtener los replays
  response = requests.get(url+'&page='+str(i))

  if response.status_code == 200:
      data = response.json()  # Convertir la respuesta a JSON

      if isinstance(data, list):  # Verificar si es una lista
          print(f" Se encontraron {len(data)} replays gen9ou en la página {i}.")

          # Descargar solo los replays
          for i, replay in enumerate(data):
              replay_id = replay['id']  # ID del replay
              replay_url = f"https://replay.pokemonshowdown.com/{replay_id}"

              # Hacer la solicitud para obtener el archivo .log del replay
              log_response = requests.get(f"{replay_url}.log")

              if log_response.status_code == 200:
                  # Guardar el archivo .log
                  log_filename = os.path.join(output_folder, f"battle_{replay_id}.log")
                  with open(log_filename, 'w', encoding='utf-8') as file:
                      file.write(log_response.text)
                  print(f"Archivo guardado: {log_filename}")
              else:
                  print(f"No se pudo descargar el .log para {replay_url}")
      else:
          print("La API devolvió un formato inesperado.")
  else:
      print("Error al acceder a la API de Showdown.")



In [ ]:
import csv
import os
import re  

def Procesar_Log_Detallado(Path_Log):
    with open(Path_Log, encoding='utf-8') as f:
        Lines = f.readlines()

    Id_Batalla = os.path.basename(Path_Log).replace(".log", "")
    Jugador1 = ""
    Jugador2 = ""
    Ganador = ""

    Pokemon_Jugador1 = []
    Pokemon_Jugador2 = []

    Turnos = []
    Turno_Actual = 0

    Activo_J1 = None
    Activo_J2 = None
    Shiny_J1 = False
    Shiny_J2 = False
    Tiempo = ""

    Acciones = {
        "p1": {"Accion": "", "Detalle": "", "Vida_Final": ""},
        "p2": {"Accion": "", "Detalle": "", "Vida_Final": ""}
    }

    Registro_Turno = False
    Jugador_Que_Actua_Primero = None

    Vivos_J1 = []
    Vivos_J2 = []

    # Diccionarios para almacenar la vida actual de cada pokemon
    Vidas_Actuales_J1 = {}
    Vidas_Actuales_J2 = {}

    for Line in Lines:
        Line = Line.strip()

        if Line.startswith("|player|p1|"):
            match = re.match(r"\|player\|p1\|([^\|]+)\|", Line)
            if match:
                Jugador1 = match.group(1)

        elif Line.startswith("|player|p2|"):
            match = re.match(r"\|player\|p2\|([^\|]+)\|", Line)
            if match:
                Jugador2 = match.group(1)

        elif Line.startswith("|poke|p1|"):
            Pokemon = Line.split("|")[3].split(',')[0]
            Pokemon_Jugador1.append(Pokemon)

        elif Line.startswith("|poke|p2|"):
            Pokemon = Line.split("|")[3].split(',')[0]
            Pokemon_Jugador2.append(Pokemon)

        elif Line.startswith("|win|"):
            Nombre_Ganador = Line.split("|")[2]
            if Nombre_Ganador == Jugador1:
                Ganador = "p1"
            elif Nombre_Ganador == Jugador2:
                Ganador = "p2"
            else:
                Ganador = "desconocido"

        elif Line.startswith("|turn|"):
            if Registro_Turno:
                # rellenar vidas si no hubo daño
                if not Acciones["p1"]["Vida_Final"] and Activo_J1:
                    Acciones["p1"]["Vida_Final"] = Vidas_Actuales_J1.get(Activo_J1, "")
                if not Acciones["p2"]["Vida_Final"] and Activo_J2:
                    Acciones["p2"]["Vida_Final"] = Vidas_Actuales_J2.get(Activo_J2, "")

                Turnos.append({
                    "Turno": Turno_Actual,
                    "Activo_Jugador1": Activo_J1,
                    "Activo_Jugador2": Activo_J2,
                    "Accion_P1": Acciones["p1"]["Accion"],
                    "Detalle_P1": Acciones["p1"]["Detalle"],
                    "Vida_P1": Acciones["p1"]["Vida_Final"],
                    "Shiny_P1": Shiny_J1,
                    "Accion_P2": Acciones["p2"]["Accion"],
                    "Detalle_P2": Acciones["p2"]["Detalle"],
                    "Vida_P2": Acciones["p2"]["Vida_Final"],
                    "Shiny_P2": Shiny_J2,
                    "Primera_Accion": Jugador_Que_Actua_Primero,
                    "Tiempo": Tiempo,
                    "Vivos_P1": len(Vivos_J1),
                    "Vivos_P2": len(Vivos_J2)
                })

            Turno_Actual = int(Line.split("|")[2])
            Acciones = {
                "p1": {"Accion": "", "Detalle": "", "Vida_Final": ""},
                "p2": {"Accion": "", "Detalle": "", "Vida_Final": ""}
            }
            Jugador_Que_Actua_Primero = None
            Registro_Turno = True
            Tiempo = ""

            if Turno_Actual == 1:
                Vivos_J1 = Pokemon_Jugador1.copy()
                Vivos_J2 = Pokemon_Jugador2.copy()
                # Inicializar vidas
                Vidas_Actuales_J1 = {p: "100" for p in Pokemon_Jugador1}
                Vidas_Actuales_J2 = {p: "100" for p in Pokemon_Jugador2}

        elif "|switch|p1a:" in Line or "|drag|p1a:" in Line:
            Info_Pokemon = Line.split("|")[3]
            Activo_J1 = Info_Pokemon.split(",")[0].strip()
            Shiny_J1 = "shiny" in Info_Pokemon.lower()
            if not Jugador_Que_Actua_Primero:
                Jugador_Que_Actua_Primero = "p1"
            Acciones["p1"]["Accion"] = "cambio"
            Acciones["p1"]["Detalle"] = Activo_J1
            # asignar vida actual
            Acciones["p1"]["Vida_Final"] = Vidas_Actuales_J1.get(Activo_J1, "")

        elif "|switch|p2a:" in Line or "|drag|p2a:" in Line:
            Info_Pokemon = Line.split("|")[3]
            Activo_J2 = Info_Pokemon.split(",")[0].strip()
            Shiny_J2 = "shiny" in Info_Pokemon.lower()
            if not Jugador_Que_Actua_Primero:
                Jugador_Que_Actua_Primero = "p2"
            Acciones["p2"]["Accion"] = "cambio"
            Acciones["p2"]["Detalle"] = Activo_J2
            Acciones["p2"]["Vida_Final"] = Vidas_Actuales_J2.get(Activo_J2, "")

        elif "|move|p1a:" in Line:
            Movimiento = Line.split("|")[3]
            if not Jugador_Que_Actua_Primero:
                Jugador_Que_Actua_Primero = "p1"
            Acciones["p1"]["Accion"] = "movimiento"
            Acciones["p1"]["Detalle"] = Movimiento

        elif "|move|p2a:" in Line:
            Movimiento = Line.split("|")[3]
            if not Jugador_Que_Actua_Primero:
                Jugador_Que_Actua_Primero = "p2"
            Acciones["p2"]["Accion"] = "movimiento"
            Acciones["p2"]["Detalle"] = Movimiento

        elif "|-enditem|p1a:" in Line:
            Objeto = Line.split("|")[3]
            Acciones["p1"]["Accion"] = "objeto"
            Acciones["p1"]["Detalle"] = Objeto

        elif "|-enditem|p2a:" in Line:
            Objeto = Line.split("|")[3]
            Acciones["p2"]["Accion"] = "objeto"
            Acciones["p2"]["Detalle"] = Objeto

        elif "|-damage|p1a:" in Line:
            Vida_Info = Line.split("|")[3]
            Acciones["p1"]["Vida_Final"] = Vida_Info.split(" ")[0]
            if Activo_J1:
                Vidas_Actuales_J1[Activo_J1] = Acciones["p1"]["Vida_Final"]

        elif "|-damage|p2a:" in Line:
            Vida_Info = Line.split("|")[3]
            Acciones["p2"]["Vida_Final"] = Vida_Info.split(" ")[0]
            if Activo_J2:
                Vidas_Actuales_J2[Activo_J2] = Acciones["p2"]["Vida_Final"]

        elif Line.startswith("|faint|p1a:"):
            Nombre = Line.split(":")[1].strip()
            if Nombre in Vivos_J1:
                Vivos_J1.remove(Nombre)

        elif Line.startswith("|faint|p2a:"):
            Nombre = Line.split(":")[1].strip()
            if Nombre in Vivos_J2:
                Vivos_J2.remove(Nombre)

        elif "|-weather|" in Line:
            Tiempo = Line.split("|")[2].strip()

    if Registro_Turno:
        if not Acciones["p1"]["Vida_Final"] and Activo_J1:
            Acciones["p1"]["Vida_Final"] = Vidas_Actuales_J1.get(Activo_J1, "")
        if not Acciones["p2"]["Vida_Final"] and Activo_J2:
            Acciones["p2"]["Vida_Final"] = Vidas_Actuales_J2.get(Activo_J2, "")

        Turnos.append({
            "Turno": Turno_Actual,
            "Activo_Jugador1": Activo_J1,
            "Activo_Jugador2": Activo_J2,
            "Accion_P1": Acciones["p1"]["Accion"],
            "Detalle_P1": Acciones["p1"]["Detalle"],
            "Vida_P1": Acciones["p1"]["Vida_Final"],
            "Shiny_P1": Shiny_J1,
            "Accion_P2": Acciones["p2"]["Accion"],
            "Detalle_P2": Acciones["p2"]["Detalle"],
            "Vida_P2": Acciones["p2"]["Vida_Final"],
            "Shiny_P2": Shiny_J2,
            "Primera_Accion": Jugador_Que_Actua_Primero,
            "Tiempo": Tiempo,
            "Vivos_P1": len(Vivos_J1),
            "Vivos_P2": len(Vivos_J2)
        })

    return {
        "Id_Batalla": Id_Batalla,
        "Jugador1": Jugador1,
        "Jugador2": Jugador2,
        "Pokemon_Jugador1": Pokemon_Jugador1,
        "Pokemon_Jugador2": Pokemon_Jugador2,
        "Ganador": Ganador,
        "Turnos": Turnos
    }
def guardar_multiples_batallas_csv(Carpeta_Logs, Output_Path):
    fieldnames = [
        "Id_Batalla", "Jugador1", "Jugador2",
        "Pokemon_Jugador1", "Pokemon_Jugador2", "Ganador",
        "Turno", "Activo_Jugador1", "Activo_Jugador2",
        "Primera_Accion",
        "Accion_P1", "Detalle_P1", "Vida_P1", "Shiny_P1",
        "Accion_P2", "Detalle_P2", "Vida_P2", "Shiny_P2",
        "Tiempo", "Vivos_P1", "Vivos_P2"
    ]
    
    with open(Output_Path, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for archivo in os.listdir(Carpeta_Logs):
            if archivo.endswith(".log"):
                full_path = os.path.join(Carpeta_Logs, archivo)
                info = Procesar_Log_Detallado(full_path)

                for turno in info["Turnos"]:
                    writer.writerow({
                        "Id_Batalla": info["Id_Batalla"],
                        "Jugador1": info["Jugador1"],
                        "Jugador2": info["Jugador2"],
                        "Pokemon_Jugador1": ",".join(info["Pokemon_Jugador1"]),
                        "Pokemon_Jugador2": ",".join(info["Pokemon_Jugador2"]),
                        "Ganador": info["Ganador"],
                        "Turno": turno["Turno"],
                        "Activo_Jugador1": turno["Activo_Jugador1"],
                        "Activo_Jugador2": turno["Activo_Jugador2"],
                        "Primera_Accion": turno["Primera_Accion"],
                        "Accion_P1": turno["Accion_P1"],
                        "Detalle_P1": turno["Detalle_P1"],
                        "Vida_P1": turno["Vida_P1"],
                        "Shiny_P1": turno["Shiny_P1"],
                        "Accion_P2": turno["Accion_P2"],
                        "Detalle_P2": turno["Detalle_P2"],
                        "Vida_P2": turno["Vida_P2"],
                        "Shiny_P2": turno["Shiny_P2"],
                        "Tiempo": turno["Tiempo"],
                        "Vivos_P1": turno["Vivos_P1"],
                        "Vivos_P2": turno["Vivos_P2"]
                    })

if __name__ == "__main__":
    carpeta = "../data/batallas_ou_logs"
    salida = "../data/batallas_ou.csv"
    guardar_multiples_batallas_csv(carpeta, salida)
    print(f"CSV generado correctamente: {salida}")
